# 2.4 Búsqueda informada: Greedy Best-First Search y A*

**Asignatura:** Introducción a la Inteligencia Artificial  
**Unidad 2:** Modelado y planteamiento de problemas

## Propósito

Implementar y comparar dos estrategias de búsqueda informada:

a) **Greedy Best-First Search**

b) **A\***

La comparación se realizará sobre el problema clásico de rutas **Arad → Bucharest**, utilizando como heurística la distancia en línea recta hacia Bucharest.

> **Idea clave:** Greedy utiliza únicamente \(h(n)\), mientras que A\* utiliza \(g(n)+h(n)\).

## 1. Mapa de rutas

Utilizaremos un subconjunto del mapa de Rumania.

Los valores de las aristas representan distancias entre ciudades.

```text
Arad
├── Zerind
├── Timisoara
└── Sibiu
    ├── Fagaras
    │   └── Bucharest
    └── Rimnicu Vilcea
        ├── Craiova
        └── Pitesti
            └── Bucharest
```

El problema será:

```text
Estado inicial = Arad
Estado objetivo = Bucharest
```

In [ ]:
grafo = {
    "Arad": {
        "Zerind": 75,
        "Timisoara": 118,
        "Sibiu": 140
    },
    "Zerind": {
        "Arad": 75
    },
    "Timisoara": {
        "Arad": 118
    },
    "Sibiu": {
        "Arad": 140,
        "Fagaras": 99,
        "Rimnicu Vilcea": 80
    },
    "Fagaras": {
        "Sibiu": 99,
        "Bucharest": 211
    },
    "Rimnicu Vilcea": {
        "Sibiu": 80,
        "Craiova": 146,
        "Pitesti": 97
    },
    "Craiova": {
        "Rimnicu Vilcea": 146,
        "Pitesti": 138
    },
    "Pitesti": {
        "Rimnicu Vilcea": 97,
        "Craiova": 138,
        "Bucharest": 101
    },
    "Bucharest": {
        "Fagaras": 211,
        "Pitesti": 101
    }
}

inicio = "Arad"
objetivo = "Bucharest"

print("Estado inicial:", inicio)
print("Estado objetivo:", objetivo)

## 2. Heurística \(h(n)\)

Utilizaremos la distancia en línea recta a Bucharest:

\[
h(n)=\text{distancia estimada desde }n\text{ hasta Bucharest}
\]

Un valor menor indica que el estado parece estar más cerca del objetivo.

In [ ]:
heuristica = {
    "Arad": 366,
    "Zerind": 374,
    "Timisoara": 329,
    "Sibiu": 253,
    "Fagaras": 176,
    "Rimnicu Vilcea": 193,
    "Craiova": 160,
    "Pitesti": 100,
    "Bucharest": 0
}

for ciudad, h in heuristica.items():
    print(f"{ciudad:16s} h(n) = {h}")

## 3. Función auxiliar para reconstruir la ruta

In [ ]:
def reconstruir_camino(padres, objetivo):
    camino = []
    actual = objetivo

    while actual is not None:
        camino.append(actual)
        actual = padres.get(actual)

    camino.reverse()
    return camino

## 4. Greedy Best-First Search

Greedy selecciona el nodo con menor valor heurístico:

\[
f(n)=h(n)
\]

No considera cuánto ha costado llegar hasta el nodo actual.

In [ ]:
import heapq

def greedy_best_first(grafo, heuristica, inicio, objetivo):
    frontera = []
    heapq.heappush(frontera, (heuristica[inicio], inicio))

    padres = {inicio: None}
    costo_acumulado = {inicio: 0}
    visitados = set()
    orden = []

    while frontera:
        h_actual, actual = heapq.heappop(frontera)

        if actual in visitados:
            continue

        visitados.add(actual)
        orden.append(actual)

        if actual == objetivo:
            camino = reconstruir_camino(padres, objetivo)
            return {
                "algoritmo": "Greedy",
                "encontrado": True,
                "camino": camino,
                "costo": costo_acumulado[objetivo],
                "orden": orden,
                "nodos_expandidos": len(orden)
            }

        for vecino, costo in grafo[actual].items():
            if vecino not in visitados and vecino not in padres:
                padres[vecino] = actual
                costo_acumulado[vecino] = costo_acumulado[actual] + costo
                heapq.heappush(
                    frontera,
                    (heuristica[vecino], vecino)
                )

    return {
        "algoritmo": "Greedy",
        "encontrado": False,
        "camino": [],
        "costo": None,
        "orden": orden,
        "nodos_expandidos": len(orden)
    }

resultado_greedy = greedy_best_first(
    grafo, heuristica, inicio, objetivo
)

resultado_greedy

### Resultado esperado

Greedy prioriza los estados que parecen estar más cerca de Bucharest y encuentra:

```text
Arad → Sibiu → Fagaras → Bucharest
```

Costo total:

\[
140+99+211=450
\]

La solución es válida, pero no es la de menor costo.

## 5. A*

A\* utiliza:

\[
f(n)=g(n)+h(n)
\]

donde:

\[
g(n)=\text{costo acumulado desde el inicio}
\]

y:

\[
h(n)=\text{estimación del costo restante}
\]

Por tanto, A\* considera simultáneamente lo que ya ha costado avanzar y lo que estima que falta.

In [ ]:
def a_star(grafo, heuristica, inicio, objetivo):
    frontera = []
    heapq.heappush(
        frontera,
        (heuristica[inicio], 0, inicio)
    )

    padres = {inicio: None}
    g_costos = {inicio: 0}
    cerrados = set()
    orden = []
    detalle = []

    while frontera:
        f_actual, g_actual, actual = heapq.heappop(frontera)

        if actual in cerrados:
            continue

        cerrados.add(actual)
        orden.append(actual)
        detalle.append({
            "Nodo": actual,
            "g(n)": g_actual,
            "h(n)": heuristica[actual],
            "f(n)": g_actual + heuristica[actual]
        })

        if actual == objetivo:
            camino = reconstruir_camino(padres, objetivo)
            return {
                "algoritmo": "A*",
                "encontrado": True,
                "camino": camino,
                "costo": g_costos[objetivo],
                "orden": orden,
                "nodos_expandidos": len(orden),
                "detalle": detalle
            }

        for vecino, costo in grafo[actual].items():
            nuevo_g = g_costos[actual] + costo

            if vecino in cerrados:
                continue

            if vecino not in g_costos or nuevo_g < g_costos[vecino]:
                g_costos[vecino] = nuevo_g
                padres[vecino] = actual
                nuevo_f = nuevo_g + heuristica[vecino]

                heapq.heappush(
                    frontera,
                    (nuevo_f, nuevo_g, vecino)
                )

    return {
        "algoritmo": "A*",
        "encontrado": False,
        "camino": [],
        "costo": None,
        "orden": orden,
        "nodos_expandidos": len(orden),
        "detalle": detalle
    }

resultado_astar = a_star(
    grafo, heuristica, inicio, objetivo
)

resultado_astar

### Resultado esperado

A\* encuentra:

```text
Arad → Sibiu → Rimnicu Vilcea → Pitesti → Bucharest
```

Costo total:

\[
140+80+97+101=418
\]

En este ejemplo, A\* obtiene una solución de menor costo que Greedy.

## 6. Valores \(g(n)\), \(h(n)\) y \(f(n)\)

La siguiente tabla permite observar cómo A\* evalúa los nodos que expande.

In [ ]:
import pandas as pd

detalle_astar = pd.DataFrame(resultado_astar["detalle"])
detalle_astar

## 7. Comparación experimental

In [ ]:
comparacion = pd.DataFrame([
    {
        "Algoritmo": "Greedy",
        "Función": "h(n)",
        "Camino": " → ".join(resultado_greedy["camino"]),
        "Costo total": resultado_greedy["costo"],
        "Orden de expansión": " → ".join(resultado_greedy["orden"]),
        "Nodos expandidos": resultado_greedy["nodos_expandidos"]
    },
    {
        "Algoritmo": "A*",
        "Función": "g(n) + h(n)",
        "Camino": " → ".join(resultado_astar["camino"]),
        "Costo total": resultado_astar["costo"],
        "Orden de expansión": " → ".join(resultado_astar["orden"]),
        "Nodos expandidos": resultado_astar["nodos_expandidos"]
    }
])

comparacion

## 8. Verificación de admisibilidad de la heurística

Una heurística es admisible si:

\[
h(n)\leq h^*(n)
\]

donde \(h^*(n)\) es el costo óptimo real desde \(n\) hasta el objetivo.

Para comprobarlo, calcularemos el costo real mínimo desde cada ciudad a Bucharest mediante Dijkstra.

In [ ]:
def dijkstra_desde_objetivo(grafo, objetivo):
    distancias = {objetivo: 0}
    frontera = [(0, objetivo)]

    while frontera:
        distancia, actual = heapq.heappop(frontera)

        if distancia > distancias[actual]:
            continue

        for vecino, costo in grafo[actual].items():
            nueva = distancia + costo

            if vecino not in distancias or nueva < distancias[vecino]:
                distancias[vecino] = nueva
                heapq.heappush(frontera, (nueva, vecino))

    return distancias

costos_reales = dijkstra_desde_objetivo(grafo, objetivo)

filas = []
for ciudad in heuristica:
    real = costos_reales.get(ciudad)
    h = heuristica[ciudad]

    filas.append({
        "Ciudad": ciudad,
        "h(n)": h,
        "Costo real óptimo h*(n)": real,
        "¿Admisible?": h <= real if real is not None else None
    })

pd.DataFrame(filas)

## 9. Experimenta con una heurística menos informativa

Una heurística puede ser admisible y, al mismo tiempo, aportar poca información.

Prueba:

\[
h(n)=0
\]

para todos los estados.

In [ ]:
heuristica_cero = {ciudad: 0 for ciudad in grafo}

resultado_astar_h0 = a_star(
    grafo, heuristica_cero, inicio, objetivo
)

print("Camino:", " → ".join(resultado_astar_h0["camino"]))
print("Costo:", resultado_astar_h0["costo"])
print("Orden de expansión:", " → ".join(resultado_astar_h0["orden"]))
print("Nodos expandidos:", resultado_astar_h0["nodos_expandidos"])

### Pregunta de análisis

¿Qué cambia cuando \(h(n)=0\)?

Observa especialmente:

a) La ruta encontrada

b) El costo final

c) El número de nodos expandidos

d) La capacidad de la heurística para orientar la búsqueda

## 10. Experimenta con una heurística que sobreestima

Modifica algunos valores para crear una heurística **no admisible**.

Por ejemplo, aumenta artificialmente el valor de un estado que forma parte de la ruta óptima.

> El objetivo es observar que una heurística mal diseñada puede cambiar el comportamiento de A\* y comprometer sus garantías teóricas.

In [ ]:
heuristica_sobreestimada = heuristica.copy()

# Ejemplo experimental:
heuristica_sobreestimada["Pitesti"] = 500

resultado_astar_mala = a_star(
    grafo, heuristica_sobreestimada, inicio, objetivo
)

print("Camino:", " → ".join(resultado_astar_mala["camino"]))
print("Costo:", resultado_astar_mala["costo"])
print("Orden de expansión:", " → ".join(resultado_astar_mala["orden"]))

## 11. Actividad experimental

Realiza al menos **dos modificaciones** y registra los resultados:

a) Cambia valores de la heurística manteniendo la admisibilidad

b) Utiliza \(h(n)=0\)

c) Sobreestima uno o más estados

d) Modifica uno de los costos de las rutas

e) Cambia el estado inicial

Para cada experimento compara:

```text
Ruta encontrada
Costo total
Orden de expansión
Nodos expandidos
```

---

## 12. Preguntas de análisis

a) ¿Por qué Greedy puede llegar rápidamente al objetivo y aun así obtener una ruta más costosa?

b) ¿Qué información adicional utiliza A\* respecto de Greedy?

c) ¿Qué función desempeña \(g(n)\)?

d) ¿Qué función desempeña \(h(n)\)?

e) ¿Qué ocurre con A\* cuando \(h(n)=0\)?

f) ¿Por qué una heurística que sobreestima puede ser problemática?

g) ¿Una heurística admisible garantiza que se expandan pocos nodos?

h) ¿Qué características debería tener una buena heurística?

---

## Conclusión

> **Greedy utiliza únicamente la estimación restante, mientras que A\* equilibra el costo recorrido y la estimación hacia el objetivo.**

La calidad de la heurística influye directamente en la eficiencia de la búsqueda.

## Referencias

Russell, S. J., & Norvig, P. (2021). *Artificial Intelligence: A Modern Approach* (4th ed.). Pearson.

Poole, D. L., & Mackworth, A. K. (2023). *Artificial Intelligence: Foundations of Computational Agents* (3rd ed.). Cambridge University Press.